# Notebook 2 — SAR Preprocessing
**Inputs:** Sentinel-1 GRD scenes via Element84 STAC  
**Outputs:** `data/processed/sar/YYYY-MM_VV.tif` and `YYYY-MM_VH.tif` (Cloud-Optimized GeoTIFFs)  
**Run after:** Notebook 01 (config must exist)  
**Run before:** Notebook 03 (flood detection reads these outputs)

## Steps
1. Load Sentinel-1 VV/VH scenes per month via stackstac (lazy/COG — no full download)
2. Median composite per month to reduce speckle across overlapping passes
3. Lee speckle filter on the composite
4. Convert linear backscatter → dB
5. Save monthly COGs to `data/processed/sar/`
6. Build 3-month dry-season baseline and save to `data/processed/sar/baseline_VV.tif`

## Strengths & Limitations
**Strengths:**
- COG-based loading: only pixels covering the AOI are downloaded — efficient over large areas
- Monthly median composite suppresses transient speckle better than single-pass filtering
- VH polarization is more sensitive to volume scattering (vegetation/flood interaction)

**Limitations:**
- Sentinel-1 GRD is not terrain-corrected by default; steep slopes in Virunga/Ruwenzori create layover and shadow artifacts that persist into the flood mask
- Lee filter window (7×7) smooths fine-scale flood boundaries — smaller inundations (<3 pixels) may be lost
- Median compositing can suppress a real flood signal if the flood only occurred during 1 of several passes in the month

In [ ]:
# Clone repo (Colab only — skipped if already present)
import os
if not os.path.exists('Floodmaps'):
    !git clone https://github.com/trevmon28/Floodmaps.git
os.chdir('Floodmaps')
print('Working directory:', os.getcwd())

In [ ]:
# Install packages (skip if already installed)
import subprocess, sys
packages = [
    'rasterio', 'rioxarray', 'xarray', 'dask[distributed]',
    'stackstac', 'pystac-client', 'geopandas', 'shapely',
    'scipy', 'scikit-image', 'numpy', 'pyyaml', 'pyproj', 'tqdm'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages)
print('Packages ready.')

In [ ]:
# Imports and config
import yaml
import numpy as np
import xarray as xr
import rioxarray
import rasterio
from rasterio.crs import CRS
from rasterio.shutil import copy as rio_copy
import stackstac
import pystac_client
import pandas as pd
from shapely.geometry import box
from scipy.ndimage import uniform_filter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

with open('config/config.yaml') as f:
    cfg = yaml.safe_load(f)

b = cfg['aoi']['bbox']
BBOX = [b['west'], b['south'], b['east'], b['north']]
OUTPUT_CRS = cfg['aoi']['output_crs']
PROCESSED_DIR = 'data/processed/sar'
BASELINE_MONTHS = cfg['processing']['baseline_months']
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f'AOI bbox: {BBOX}')
print(f'Output CRS: {OUTPUT_CRS}')
print(f'Baseline period: first {BASELINE_MONTHS} months')

In [ ]:
# Helper functions

def monthly_date_ranges(cfg):
    start = pd.Timestamp(cfg['temporal']['start'])
    end = pd.Timestamp(cfg['temporal']['end'])
    months = pd.date_range(start, end, freq='MS')
    return [(str(m.date()), str((m + pd.offsets.MonthEnd(1)).date())) for m in months]

def to_db(arr):
    """Convert linear backscatter to dB."""
    return 10 * np.log10(np.where(arr > 0, arr, np.nan))

def lee_filter(arr, size=7):
    """Lee speckle filter."""
    img = arr.astype('float64')
    img_mean = uniform_filter(img, size)
    img_sq_mean = uniform_filter(img ** 2, size)
    img_var = img_sq_mean - img_mean ** 2
    overall_var = np.nanvar(img)
    weights = img_var / (img_var + overall_var + 1e-10)
    return img_mean + weights * (img - img_mean)

def write_cog(array, profile, output_path):
    """Write array as Cloud-Optimized GeoTIFF with overviews."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    tmp = output_path + '.tmp.tif'
    profile.update(driver='GTiff', compress='deflate', tiled=True,
                   blockxsize=512, blockysize=512, dtype='float32', count=1)
    with rasterio.open(tmp, 'w', **profile) as dst:
        dst.write(array.astype('float32'), 1)
    with rasterio.open(tmp, 'r+') as dst:
        dst.build_overviews([2, 4, 8, 16], rasterio.enums.Resampling.average)
        dst.update_tags(ns='rio_overview', resampling='average')
    rio_copy(tmp, output_path, driver='GTiff', copy_src_overviews=True,
             compress='deflate', tiled=True, blockxsize=512, blockysize=512)
    os.remove(tmp)
    print(f'  Saved: {output_path}')

print('Helper functions defined.')

In [ ]:
# Connect to Element84 STAC catalog
catalog = pystac_client.Client.open('https://earth-search.aws.element84.com/v1')
date_ranges = monthly_date_ranges(cfg)
print(f'{len(date_ranges)} monthly windows to process')
print(f'First: {date_ranges[0]}  |  Last: {date_ranges[-1]}')

In [ ]:
# Process one month as a test (change month_idx to process others)
# Set process_all = True to run the full Jan 2025 - Apr 2026 pipeline
process_all = False
month_idx = 0  # 0 = Jan 2025

months_to_run = date_ranges if process_all else [date_ranges[month_idx]]
monthly_vv = {}  # store processed arrays for baseline computation

for start, end in tqdm(months_to_run, desc='Processing months'):
    month_str = start[:7]  # YYYY-MM
    out_vv = f'{PROCESSED_DIR}/{month_str}_VV.tif'
    out_vh = f'{PROCESSED_DIR}/{month_str}_VH.tif'

    if os.path.exists(out_vv) and os.path.exists(out_vh):
        print(f'  {month_str} already processed — skipping')
        continue

    print(f'\nSearching Sentinel-1 for {month_str}...')
    results = catalog.search(
        collections=['sentinel-1-grd'],
        bbox=BBOX,
        datetime=f'{start}/{end}',
    )
    items = list(results.items())
    print(f'  Found {len(items)} scenes')

    if not items:
        print(f'  No scenes found for {month_str} — skipping')
        continue

    # Load VV and VH as lazy dask arrays via stackstac
    stack = stackstac.stack(
        items,
        assets=['vv', 'vh'],
        bounds_latlon=BBOX,
        epsg=int(OUTPUT_CRS.split(':')[1]),
        resolution=20,
        dtype='float32',
        rescale=False,
    )
    print(f'  Stack shape: {stack.shape} (time, band, y, x)')

    # Monthly median composite — reduces speckle across overlapping passes
    composite = stack.median(dim='time').compute()

    for band_name, band_idx in [('VV', 0), ('VH', 1)]:
        arr = composite.isel(band=band_idx).values
        arr = lee_filter(arr)          # speckle filter
        arr = to_db(arr)               # linear → dB

        profile = {
            'crs': OUTPUT_CRS,
            'transform': composite.rio.transform(),
            'width': arr.shape[1],
            'height': arr.shape[0],
            'nodata': np.nan,
        }
        out_path = f'{PROCESSED_DIR}/{month_str}_{band_name}.tif'
        write_cog(arr, profile, out_path)

        if band_name == 'VV':
            monthly_vv[month_str] = arr  # keep in memory for baseline

print('\nMonth processing complete.')

In [ ]:
# Build dry-season baseline from first N months (VV band)
# Baseline = median of BASELINE_MONTHS months — represents non-flood backscatter
baseline_path = f'{PROCESSED_DIR}/baseline_VV.tif'

if not os.path.exists(baseline_path):
    baseline_files = sorted([
        f for f in os.listdir(PROCESSED_DIR) if f.endswith('_VV.tif')
    ])[:BASELINE_MONTHS]

    if len(baseline_files) < BASELINE_MONTHS:
        print(f'Only {len(baseline_files)} months available — run process_all=True first')
    else:
        arrays = []
        profile = None
        for fname in baseline_files:
            with rasterio.open(f'{PROCESSED_DIR}/{fname}') as src:
                arrays.append(src.read(1).astype('float32'))
                if profile is None:
                    profile = src.profile.copy()

        baseline = np.nanmedian(np.stack(arrays, axis=0), axis=0)
        write_cog(baseline, profile, baseline_path)
        print(f'Baseline built from: {baseline_files}')
else:
    print(f'Baseline already exists: {baseline_path}')

In [ ]:
# Quick visual check of one processed month
import matplotlib.pyplot as plt

sample_file = sorted([f for f in os.listdir(PROCESSED_DIR) if f.endswith('_VV.tif')])
if sample_file:
    with rasterio.open(f'{PROCESSED_DIR}/{sample_file[0]}') as src:
        vv_db = src.read(1)

    plt.figure(figsize=(12, 8))
    plt.imshow(vv_db, cmap='gray', vmin=-25, vmax=0)
    plt.colorbar(label='Backscatter (dB)')
    plt.title(f'Sentinel-1 VV — {sample_file[0]} (dB)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Value range: {np.nanmin(vv_db):.1f} to {np.nanmax(vv_db):.1f} dB')
    print('Water surfaces typically appear at -20 to -15 dB')
    print('Vegetation/urban typically -10 to -5 dB')
else:
    print('No processed files yet — run the processing cell above first.')